# White Blood Cell Classification

A training and evaluation pipeline for classifying microscopy images into 13 white blood cell types. The pipeline uses a pretrained ConvNeXt-Base model, adaptive median denoising, class-weighted loss, two-phase fine-tuning, and deterministic eight-view test-time augmentation.

The dataset and trained checkpoints are not included. See the repository README for the expected data layout and reported results.

## 1. Setup

In [ ]:
import gc
import os
import random
import warnings
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device: {DEVICE}")

## 2. Configuration and data layout

In [ ]:
def find_project_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "notebooks").exists() and (candidate / "README.md").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


def ensure_png(name):
    name = str(name)
    return name if name.lower().endswith(".png") else name + ".png"


PROJECT_ROOT = find_project_root()
DATA_ROOT = Path(os.getenv("WBC_DATA_ROOT", PROJECT_ROOT / "data")).expanduser().resolve()
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
TRAIN_CSV = DATA_ROOT / "train_metadata.csv"
TEST_CSV = DATA_ROOT / "test_metadata.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

required = [TRAIN_DIR, TEST_DIR, TRAIN_CSV, TEST_CSV]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Dataset files are missing. Follow the data layout in README.md or set "
        "WBC_DATA_ROOT. Missing:\n- " + "\n- ".join(missing)
    )

MODEL_NAME = "convnext_base.fb_in22k_ft_in1k"
IMG_SIZE = 300
NUM_CLASSES = 13
BATCH_SIZE = 32
VAL_FRAC = 0.2
LABEL_SMOOTH = 0.1

P1_EPOCHS = 5
P1_LR = 2e-3
P2_EPOCHS = 30
P2_LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

P1_BEST_CKPT = OUTPUT_DIR / "convnext_base_phase1_best.pt"
P1_LATEST_CKPT = OUTPUT_DIR / "convnext_base_phase1_latest.pt"
BEST_CKPT = OUTPUT_DIR / "convnext_base_best.pt"
LATEST_CKPT = OUTPUT_DIR / "convnext_base_latest.pt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Model: {MODEL_NAME}")

## 3. Metadata, split, and class weights

In [ ]:
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw = pd.read_csv(TEST_CSV)

required_train_columns = {"ID", "label"}
required_test_columns = {"ID"}
if not required_train_columns.issubset(df_train_raw.columns):
    raise ValueError(f"train_metadata.csv must contain {sorted(required_train_columns)}")
if not required_test_columns.issubset(df_test_raw.columns):
    raise ValueError(f"test_metadata.csv must contain {sorted(required_test_columns)}")

df = df_train_raw.copy()
df["file_path"] = df["ID"].apply(lambda value: str(TRAIN_DIR / ensure_png(value)))

unique_labels = sorted(df["label"].unique())
LABEL_MAP = {label: index for index, label in enumerate(unique_labels)}
IDX2LABEL = {index: label for label, index in LABEL_MAP.items()}
NUM_CLASSES = len(LABEL_MAP)
df["label_idx"] = df["label"].map(LABEL_MAP)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, df["label_idx"]))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)

class_counts = np.bincount(df_train["label_idx"], minlength=NUM_CLASSES).astype(np.float32)
class_weights = 1.0 / np.sqrt(class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print(f"Classes ({NUM_CLASSES}): {list(LABEL_MAP)}")
print(f"Train: {len(df_train):,} | Validation: {len(df_val):,}")
for index in range(NUM_CLASSES):
    print(f"{IDX2LABEL[index]:>4s}: {int(class_counts[index]):>5d} samples, weight={class_weights[index]:.4f}")

## 4. Adaptive median denoising

In [ ]:
def impulse_noise_score(image_rgb):
    median = cv2.medianBlur(image_rgb, 3)
    difference = np.abs(image_rgb.astype(np.float32) - median.astype(np.float32))
    return float(difference.mean())


def adaptive_median_filter(image_rgb, score):
    if score < 10:
        return image_rgb
    if score < 20:
        kernel_size = 3
    elif score < 40:
        kernel_size = 5
    else:
        kernel_size = 7
    return cv2.medianBlur(image_rgb, kernel_size)


def denoise(image_rgb):
    score = impulse_noise_score(image_rgb)
    return adaptive_median_filter(image_rgb, score), score

In [ ]:
N_PER_LEVEL = 2
N_SCAN = min(600, len(df))
sample_paths = df.sample(N_SCAN, random_state=SEED + 6)["file_path"].tolist()

buckets = {
    "No filter (score < 10)": [],
    "3x3 median (10 <= score < 20)": [],
    "5x5 median (20 <= score < 40)": [],
    "7x7 median (score >= 40)": [],
}

for path in tqdm(sample_paths, desc="Scanning noise", leave=False):
    image = np.array(Image.open(path).convert("RGB"))
    score = impulse_noise_score(image)
    if score < 10:
        key = list(buckets)[0]
    elif score < 20:
        key = list(buckets)[1]
    elif score < 40:
        key = list(buckets)[2]
    else:
        key = list(buckets)[3]
    buckets[key].append((score, image))

fig, axes = plt.subplots(len(buckets), N_PER_LEVEL * 2, figsize=(10, 11))
for row, (label, items) in enumerate(buckets.items()):
    items.sort(key=lambda item: item[0])
    midpoint = len(items) // 2
    picks = items[max(0, midpoint - N_PER_LEVEL // 2):][:N_PER_LEVEL]
    axes[row, 0].set_ylabel(label, fontsize=8, rotation=0, labelpad=78, va="center")

    for column in range(N_PER_LEVEL):
        before = axes[row, column * 2]
        after = axes[row, column * 2 + 1]
        if column >= len(picks):
            before.axis("off")
            after.axis("off")
            continue
        score, image = picks[column]
        filtered, _ = denoise(image)
        before.imshow(image)
        before.set_title(f"Before (score={score:.1f})", fontsize=8)
        after.imshow(filtered)
        after.set_title("After", fontsize=8)
        before.axis("off")
        after.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "denoising_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Augmentation and deterministic eight-view TTA

In [ ]:
class DihedralTransform(A.ImageOnlyTransform):
    """Apply one of the eight symmetries of a square."""

    def __init__(self, rotations=0, reflect=False, p=1.0):
        super().__init__(p=p)
        self.rotations = rotations
        self.reflect = reflect

    def apply(self, image, **params):
        transformed = np.rot90(image, self.rotations)
        if self.reflect:
            transformed = np.fliplr(transformed)
        return np.ascontiguousarray(transformed)


def train_transforms():
    return A.Compose([
        A.CenterCrop(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])


def validation_transforms():
    return A.Compose([
        A.CenterCrop(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])


def tta_transforms():
    transforms = []
    for rotations in range(4):
        for reflect in (False, True):
            transforms.append(A.Compose([
                A.CenterCrop(IMG_SIZE, IMG_SIZE),
                DihedralTransform(rotations=rotations, reflect=reflect, p=1.0),
                A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
                ToTensorV2(),
            ]))
    return transforms

print(f"TTA views: {len(tta_transforms())}")

## 6. Dataset and data loaders

In [ ]:
class WBCDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = row["file_path"]
        try:
            image = np.array(Image.open(image_path).convert("RGB"))
        except Exception as error:
            raise RuntimeError(f"Could not load {image_path}") from error

        image, _ = denoise(image)
        image = self.transform(image=image)["image"]
        label = int(row["label_idx"]) if "label_idx" in row else -1
        return image, label


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


train_dataset = WBCDataset(df_train, train_transforms())
val_dataset = WBCDataset(df_val, validation_transforms())
train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)

print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

## 7. Model and training utilities

In [ ]:
def model_core(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def build_model():
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    return model.to(DEVICE)


def freeze_backbone(model):
    core = model_core(model)
    for parameter in core.parameters():
        parameter.requires_grad = False
    for parameter in core.get_classifier().parameters():
        parameter.requires_grad = True


def unfreeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = True


def count_trainable(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad) / 1e6


def load_weights(model, state_dict):
    cleaned = {key.removeprefix("module."): value for key, value in state_dict.items()}
    model_core(model).load_state_dict(cleaned)


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1, patience_counter):
    torch.save({
        "model_state": model_core(model).state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "epoch": epoch,
        "best_f1": best_f1,
        "patience_counter": patience_counter,
        "label_map": LABEL_MAP,
    }, path)


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_labels, all_predictions = [], []

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=PIN_MEMORY)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(logits.argmax(dim=1).detach().cpu().numpy())

    loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_predictions, average="macro", zero_division=0)
    per_class_f1 = f1_score(
        all_labels,
        all_predictions,
        average=None,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )
    return loss, macro_f1, per_class_f1


def train_phase(model, optimizer, scheduler, criterion, epochs, best_path, latest_path, phase_name):
    best_f1 = -np.inf
    patience_counter = 0
    start_epoch = 1

    if latest_path.exists():
        checkpoint = torch.load(latest_path, map_location=DEVICE)
        load_weights(model, checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_f1 = checkpoint["best_f1"]
        patience_counter = checkpoint["patience_counter"]
        print(f"Resuming {phase_name} from epoch {start_epoch}")

    try:
        for epoch in range(start_epoch, epochs + 1):
            train_loss, train_f1, _ = run_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_f1, per_class = run_epoch(model, val_loader, criterion)
            scheduler.step()

            print(
                f"[{phase_name}] {epoch:02d}/{epochs} "
                f"train loss={train_loss:.4f} F1={train_f1:.4f} | "
                f"val loss={val_loss:.4f} F1={val_f1:.4f}"
            )
            for index, score in enumerate(per_class):
                print(f"  {IDX2LABEL[index]:>4s}: {score:.4f}")

            if val_f1 > best_f1:
                best_f1 = val_f1
                patience_counter = 0
                torch.save({
                    "model_state": model_core(model).state_dict(),
                    "label_map": LABEL_MAP,
                    "best_f1": best_f1,
                }, best_path)
                print(f"  New best validation macro-F1: {best_f1:.4f}")
            else:
                patience_counter += 1
                print(f"  No improvement ({patience_counter}/{PATIENCE})")

            save_checkpoint(
                latest_path,
                model,
                optimizer,
                scheduler,
                epoch,
                best_f1,
                patience_counter,
            )

            if patience_counter >= PATIENCE:
                print("Early stopping.")
                break
    except KeyboardInterrupt:
        print("Training interrupted; the latest completed epoch is already checkpointed.")

    return best_f1

## 8. Phase 1: train the classification head

In [ ]:
model = build_model()
freeze_backbone(model)
print(f"Trainable parameters: {count_trainable(model):.3f}M")

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE),
    label_smoothing=LABEL_SMOOTH,
)
phase1_optimizer = torch.optim.AdamW(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=P1_LR,
    weight_decay=WEIGHT_DECAY,
)
phase1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    phase1_optimizer,
    T_max=P1_EPOCHS,
)

phase1_best_f1 = train_phase(
    model,
    phase1_optimizer,
    phase1_scheduler,
    criterion,
    P1_EPOCHS,
    P1_BEST_CKPT,
    P1_LATEST_CKPT,
    "Phase 1",
)
print(f"Phase 1 best validation macro-F1: {phase1_best_f1:.4f}")

## 9. Phase 2: fine-tune the full network

In [ ]:
if not P1_BEST_CKPT.exists():
    raise FileNotFoundError("Run Phase 1 before starting Phase 2.")

checkpoint = torch.load(P1_BEST_CKPT, map_location=DEVICE)
load_weights(model, checkpoint["model_state"])
unfreeze_all(model)

core = model_core(model)
if hasattr(core, "set_grad_checkpointing"):
    core.set_grad_checkpointing(enable=True)

print(f"Trainable parameters: {count_trainable(model):.2f}M")

phase2_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=P2_LR,
    weight_decay=WEIGHT_DECAY,
)
phase2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    phase2_optimizer,
    T_max=P2_EPOCHS,
)

phase2_best_f1 = train_phase(
    model,
    phase2_optimizer,
    phase2_scheduler,
    criterion,
    P2_EPOCHS,
    BEST_CKPT,
    LATEST_CKPT,
    "Phase 2",
)
print(f"Phase 2 best validation macro-F1: {phase2_best_f1:.4f}")

## 10. Deterministic test-time augmentation

In [ ]:
def predict_probabilities(model, frame, transforms):
    summed_probabilities = None
    for transform in tqdm(transforms, desc="TTA"):
        dataset = WBCDataset(frame, transform)
        loader = make_loader(dataset)
        probabilities = []

        with torch.no_grad():
            for images, _ in loader:
                logits = model(images.to(DEVICE, non_blocking=PIN_MEMORY))
                probabilities.append(torch.softmax(logits, dim=1).cpu())

        probabilities = torch.cat(probabilities)
        summed_probabilities = (
            probabilities
            if summed_probabilities is None
            else summed_probabilities + probabilities
        )

    return summed_probabilities / len(transforms)


if not BEST_CKPT.exists():
    raise FileNotFoundError("Run Phase 2 before evaluation.")

checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
load_weights(model, checkpoint["model_state"])
model.eval()

val_probabilities = predict_probabilities(model, df_val, tta_transforms())
val_predictions = val_probabilities.argmax(dim=1).numpy()
val_labels = df_val["label_idx"].to_numpy()

val_macro_f1 = f1_score(val_labels, val_predictions, average="macro", zero_division=0)
val_per_class_f1 = f1_score(
    val_labels,
    val_predictions,
    average=None,
    labels=list(range(NUM_CLASSES)),
    zero_division=0,
)

print(f"TTA validation macro-F1: {val_macro_f1:.4f}")
for index, score in enumerate(val_per_class_f1):
    print(f"  {IDX2LABEL[index]:>4s}: {score:.4f}")

## 11. Confusion matrix

In [ ]:
class_names = [IDX2LABEL[index] for index in range(NUM_CLASSES)]
confusion = confusion_matrix(val_labels, val_predictions, labels=list(range(NUM_CLASSES)))
row_totals = confusion.sum(axis=1, keepdims=True)
confusion_normalized = np.divide(
    confusion,
    row_totals,
    out=np.zeros_like(confusion, dtype=float),
    where=row_totals != 0,
)

fig, axis = plt.subplots(figsize=(11, 9))
sns.heatmap(
    confusion_normalized,
    annot=confusion,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.4,
    linecolor="lightgrey",
    vmin=0,
    vmax=1,
    ax=axis,
)
axis.set_xlabel("Predicted")
axis.set_ylabel("True")
axis.set_title(f"Confusion matrix - TTA validation macro-F1={val_macro_f1:.4f}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Generate a test submission

In [ ]:
df_test = df_test_raw.copy()
df_test["file_path"] = df_test["ID"].apply(lambda value: str(TEST_DIR / ensure_png(value)))

test_probabilities = predict_probabilities(model, df_test, tta_transforms())
test_predictions = test_probabilities.argmax(dim=1).numpy()

df_test["label"] = [IDX2LABEL[prediction] for prediction in test_predictions]
submission = df_test[["ID", "label"]]
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(submission.head())
print(submission["label"].value_counts())